### Import all packages need in the cell below

In [ ]:
using JuMP
using Ipopt
using HiGHS
using Plots

# Problem Statement

Plot the graph of the function sin(x) over the interval $[-\pi/4, 3\pi/4]$

In [ ]:
x = range(-pi/4, 3pi/4, length=100)
y = sin.(x)
plot(x, y, label="sin(x)", title="Plot of sin(x)", xlabel="x", ylabel="sin(x)")

Plot the graph of the function $x\cdot sin(x)$ over the interval $[-10\pi, 10\pi]$

In [ ]:
x = range(-10pi, 10pi, length=1000)
y = x .* sin.(x)
plot(x, y, label="x*sin(x)", title="Plot of x*sin(x)", xlabel="x", ylabel="x*sin(x)")

# Problem Statement

Solve the Cylinder Problem considering the following data:

* N: 10
* $c_1$: 2
* $c_2$: 0.5
  

In [ ]:
N = 10
c1 = 2
c2 = 0.5

model = Model(Ipopt.Optimizer)
set_silent(model)

@variable(model, r >= 0, start = 1.0)
@variable(model, h >= 0, start = 1.0)

@NLobjective(model, Max, N * pi * r^2 * h - c1 * pi * r^2 - c2 * pi * r^2 * h * (pi * r^2 + 2 * pi * r * h))

optimize!(model)

println("Status: ", termination_status(model))
println("Radius (r): ", value(r))
println("Height (h): ", value(h))
println("Objective (Profit): ", objective_value(model))

# Problem Statement

Solve the Awning Problem considering the following data:

* h: 2
* w: 3
* initial guess $(x,y) = (1.0, 1.0)$
  

In [ ]:
h_val = 2
w_val = 3

model = Model(Ipopt.Optimizer)
set_silent(model)

@variable(model, x >= 0, start = 1.0)
@variable(model, y >= 0, start = 1.0)

@NLobjective(model, Min, sqrt(x^2 + y^2))
@NLconstraint(model, box, y - (y/x)*w_val >= h_val)

optimize!(model)

println("Status: ", termination_status(model))
println("x: ", value(x))
println("y: ", value(y))
println("Min Length: ", objective_value(model))

# Problem Statement

Solve the Packing Problem

In [ ]:
model = Model(Ipopt.Optimizer)
set_silent(model)

@variable(model, h >= 0, start = 1.0)
@variable(model, w >= 0, start = 1.0)
@variable(model, d >= 0, start = 1.0)

@objective(model, Max, h * w * d)
@constraint(model, cardboard, 2*w*h + 2*d*h + 6*w*d <= 60)

optimize!(model)

println("Status: ", termination_status(model))
println("Height (h): ", value(h))
println("Width (w): ", value(w))
println("Depth (d): ", value(d))
println("Max Volume: ", objective_value(model))

# Problem Statement

Solve the 3-bus Optimal Power Flow Problem with following data:

![image.png](attachment:dea684d8-257b-4a4e-a3f9-f6993fad9896.png)

In [ ]:
yl = 9.95037190209989
al = -1.47112767430373

model = Model(Ipopt.Optimizer)
set_silent(model)

@variable(model, 0 <= p1 <= 3)
@variable(model, 0 <= p2 <= 0.8)
@variable(model, -2 <= q1 <= 2)
@variable(model, -2 <= q2 <= 2)
@variable(model, 0.95 <= v1 <= 1.1, start = 1.0)
@variable(model, 0.95 <= v2 <= 1.1, start = 1.0)
@variable(model, 0.95 <= v3 <= 1.1, start = 1.0)
@variable(model, -pi <= d1 <= pi, start = 0.0)
@variable(model, -pi <= d2 <= pi, start = 0.0)
d3 = 0.0 # Reference bus

@objective(model, Min, 2*p1 + p2)

# Active Power Balance
@NLconstraint(model, bp1, p1 == yl*cos(al)*v1^2 - yl*v1*v2*cos(d1-d2-al) + yl*cos(al)*v1^2 - yl*v1*v3*cos(d1-d3-al))
@NLconstraint(model, bp2, p2 == yl*cos(al)*v2^2 - yl*v2*v1*cos(d2-d1-al) + yl*cos(al)*v2^2 - yl*v2*v3*cos(d2-d3-al))
@NLconstraint(model, bp3, -0.8 == yl*cos(al)*v3^2 - yl*v3*v1*cos(d3-d1-al) + yl*cos(al)*v3^2 - yl*v3*v2*cos(d3-d2-al))

# Reactive Power Balance
@NLconstraint(model, bq1, q1 == -yl*sin(al)*v1^2 - yl*v1*v2*sin(d1-d2-al) - yl*sin(al)*v1^2 - yl*v1*v3*sin(d1-d3-al))
@NLconstraint(model, bq2, q2 == -yl*sin(al)*v2^2 - yl*v2*v1*sin(d2-d1-al) - yl*sin(al)*v2^2 - yl*v2*v3*sin(d2-d3-al))
@NLconstraint(model, bq3, -0.6 == -yl*sin(al)*v3^2 - yl*v3*v1*sin(d3-d1-al) - yl*sin(al)*v3^2 - yl*v3*v2*sin(d3-d2-al))

# Line capacity limits
@NLconstraint(model, l12, sqrt((yl*cos(al)*v1^2 - yl*v1*v2*cos(d1-d2-al))^2 + (-yl*sin(al)*v1^2 - yl*v1*v2*sin(d1-d2-al))^2) <= 0.25)
@NLconstraint(model, l13, sqrt((yl*cos(al)*v1^2 - yl*v1*v3*cos(d1-d3-al))^2 + (-yl*sin(al)*v1^2 - yl*v1*v3*sin(d1-d3-al))^2) <= 2.0)
@NLconstraint(model, l23, sqrt((yl*cos(al)*v2^2 - yl*v2*v3*cos(d2-d3-al))^2 + (-yl*sin(al)*v2^2 - yl*v2*v3*sin(d2-d3-al))^2) <= 2.0)

optimize!(model)

println("Status: ", termination_status(model))
println("p1: ", value(p1))
println("p2: ", value(p2))
println("v1: ", value(v1))
println("v2: ", value(v2))
println("v3: ", value(v3))
println("Min Cost: ", objective_value(model))

# Problem Statement

Linear Regression with 3 variables.

Consider fi tting a linear model to the following data points with three features:

|      Observation     | $x_1$    | $x_2$    | $x_3$    | Response $(y)$    |
|----------|--------------|--------------|--------------|--------------|
| 1        | 1.0           | 0.5           | 1.2           | 2.0           |
| 2        | 2.0           | 1.0           | 2.1           | 3.9           |
| 3        | 3.0           | 1.5           | 2.9           | 6.1           |
| 4        | 4.0           | 2.0           | 3.8           | 8.0           |
| 5        | 5.0           | 2.5           | 4.5           | 9.8           |

The goal is to find the model $y = \beta_0 + \beta_1x_1 + \beta_2x_2 + \beta_3x_3$ that best fits this data in the least squares sense.

In [ ]:
x1 = [1.0, 2.0, 3.0, 4.0, 5.0]
x2 = [0.5, 1.0, 1.5, 2.0, 2.5]
x3 = [1.2, 2.1, 2.9, 3.8, 4.5]
y_data = [2.0, 3.9, 6.1, 8.0, 9.8]

model = Model(Ipopt.Optimizer)
set_silent(model)

@variable(model, beta[0:3])

@objective(model, Min, sum((y_data[i] - (beta[0] + beta[1]*x1[i] + beta[2]*x2[i] + beta[3]*x3[i]))^2 for i in 1:5))

optimize!(model)

println("Status: ", termination_status(model))
for i in 0:3
    println("beta_$i: ", value(beta[i]))
end

# Problem Statement

A small engineering consulting firm has 3 senior designers available to work on the firm's 4 current projects over the next 2 weeks. Each designer has 80 hours to split among the projects, and the following table shows the manager's scoring $(0=$ nil to $100=$ perfect $)$ of the capability of each designer to contribute to each project, along with his estimate of the hours that each project will require.


|      Designer     | Project 1    | Project 2    | Project 3    | Project 4    |
|----------|--------------|--------------|--------------|--------------|
| 1        | 90           | 80           | 10           | 50           |
| 2        | 60           | 70           | 50           | 65           |
| 3        | 70           | 40           | 80           | 85           |



|     **Required:**      | Project 1    | Project 2    | Project 3    | Project 4    |
|-----------|--------------|--------------|--------------|--------------|
| **Hours** | 70           | 50           | 85           | 35           |


## Model

Let the design engineers be set $E$ with $E_{i} \; :i \in [1,2,3]$ and the projects be $P$ with  $P_{j} \; :j \in [1,2,3,4]$. We can model the problem as allocation of the number of hours $H_{ij}$ with each design engineer $E_{i}$ $\forall i $ that are being put onto the projects $P_{j}$ $\forall j $, given the $i^{th}$ engineer $E$ works on $j^{th}$ project with given score $e_{ij}$.

Let the maximum hours available with each engineer be $H_{max}$ and the required number of hours for each project $P_j$ be $R_{j} \; \forall j $. Thus, then the mathematical formulation can be made as:

$$
\text{Maximize } \sum_{i \in E} \sum_{j \in P} H_{ij} e_{ij}
$$

Subject to:

$$
\sum_{j \in P} H_{ij} \leq H_{max} \;\; \forall i \in E
$$

$$
\sum_{i \in E} H_{ij} \geq R_j \;\; \forall j \in P
$$

$$
H_{ij} \geq 0 \;\; \forall i,j
$$


## Implement

In [ ]:
scores = [90 80 10 50; 60 70 50 65; 70 40 80 85]
required_hours = [70, 50, 85, 35]
max_hours = 80

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, H[1:3, 1:4] >= 0)

@objective(model, Max, sum(H[i,j] * scores[i,j] for i in 1:3, j in 1:4))

@constraint(model, designer_limit[i in 1:3], sum(H[i,j] for j in 1:4) <= max_hours)
@constraint(model, project_requirement[j in 1:4], sum(H[i,j] for i in 1:3) >= required_hours[j])

optimize!(model)

println("Status: ", termination_status(model))
println("Total Score: ", objective_value(model))
println("Hours Allocation (H_ij):")
display(value.(H))

# Problem Statement

A dietitian is planning a meal that meets the daily nutritional requirements for calories, protein, and vitamins at a minimum cost.


|      Food Item     | Cost ($)    | Calories    | Protein (g)    | Vitamins (% Daily)    |
|----------|--------------|--------------|--------------|--------------|
| Apple        | 1              | 100           | 0.5           | 2            |
| Bread        | 0.50           | 200           | 4             | 0            |
| Milk         | 2              | 150           | 8             | 10           |
| Egg          | 0.30           | 70            | 6             | 0            |

Daily nutritional requirements: 500 calories, 50g protein, 100% vitamins.

Define decision variables: $y_1$ for Apples, $y_2$ for Bread, $y_3$ for Milk, $y_4$ for Eggs.  
$y_i$ represents the quantity of each food item.

$$
\begin{aligned}
\text{Minimize} \quad & y_1 + 0.5y_2 + 2y_3 + 0.3y_4 \\
\text{Subject to} \quad 
& 100y_1 + 200y_2 + 150y_3 + 70y_4 \geq 500 \\
& 0.5y_1 + 4y_2 + 8y_3 + 6y_4 \geq 50 \\
& 2y_1 + 0y_2 + 10y_3 + 0y_4 \geq 100 \\
& y_1, y_2, y_3, y_4 \geq 0
\end{aligned}
$$

Ensure all dietary requirements for calories, protein, and vitamins are met.

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, y[1:4] >= 0)

@objective(model, Min, 1*y[1] + 0.5*y[2] + 2*y[3] + 0.3*y[4])

@constraint(model, calories, 100*y[1] + 200*y[2] + 150*y[3] + 70*y[4] >= 500)
@constraint(model, protein, 0.5*y[1] + 4*y[2] + 8*y[3] + 6*y[4] >= 50)
@constraint(model, vitamins, 2*y[1] + 0*y[2] + 10*y[3] + 0*y[4] >= 100)

optimize!(model)

println("Status: ", termination_status(model))
println("Apples (y1): ", value(y[1]))
println("Bread (y2): ", value(y[2]))
println("Milk (y3): ", value(y[3]))
println("Eggs (y4): ", value(y[4]))
println("Min Cost: ", objective_value(model))

# Problem Statement

Consider a hiker who needs to choose the most valuable items for a hike without overloading the backpack.

- Items: Tent (Value: $\$120$, Weight: 2kg), Stove (Value: $\$80$, Weight: 1kg), Food (Value: $\$60$, Weight: 1kg)  
- Backpack capacity: 3.5kg  

Objective: Maximize the value of items in the backpack.

Define binary decision variables: $x_1$ for Tent, $x_2$ for Stove, $x_3$ for Food.  
$x_i = 1$ if the item is chosen, and 0 otherwise.

$$
\begin{aligned}
\text{Maximize} \quad & 120x_1 + 80x_2 + 60x_3 \\
\text{Subject to} \quad 
& 2x_1 + x_2 + x_3 \leq 3.5 \\
& x_1, x_2, x_3 \in \{0,1\}
\end{aligned}
$$

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:3], Bin)

@objective(model, Max, 120*x[1] + 80*x[2] + 60*x[3])
@constraint(model, weight, 2*x[1] + x[2] + x[3] <= 3.5)

optimize!(model)

println("Status: ", termination_status(model))
println("Tent (x1): ", value(x[1]))
println("Stove (x2): ", value(x[2]))
println("Food (x3): ", value(x[3]))
println("Max Value: ", objective_value(model))